# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note**: With Croissant datasets, all entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List record sets available in the dataset by their @id
record_sets = []
for rs in metadata.record_sets:
    print(f"Record Set @id: {rs['@id']} | Name: {rs.get('name', '')}")
    record_sets.append(rs['@id'])
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    Field @id: {field['@id']}, Name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, populate the list of all available record set @ids (from above cell's output)
record_sets_ids = record_sets  # Uses populated record_sets list from previous cell
dataframes = {}

for record_set_id in record_sets_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame for RecordSet @id {record_set_id} loaded. Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for RecordSet @id {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, or grouping by key attributes in preparation for further analysis.

**Note**: Use the actual record set and field `@id`s found in section 2.

In [ ]:
# Example EDA: Replace placeholders below with the actual record set and field @ids from above

# Select a record set to analyze
example_record_set_id = record_sets_ids[0] if record_sets_ids else None  # pick first for the example

if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    # Choose a numeric field for EDA: attempt to pick one automatically
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    if not numeric_fields:
        # Try to coerce columns to numeric & find any
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_fields.append(col)
            except Exception:
                pass
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}\n")

        threshold = df[numeric_field].mean()  # arbitrary cutoff for filtering
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field if present
        group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No categorical group field found.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No suitable dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using basic plots.

In [ ]:
# Example visualization for numeric field(s)
import matplotlib.pyplot as plt
import seaborn as sns
if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} in filtered records')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if f'{numeric_field}_normalized' in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f'{numeric_field}_normalized'], bins=30, kde=True, color='orange')
        plt.title(f'Normalized Distribution of {numeric_field}')
        plt.xlabel(f'{numeric_field}_normalized')
        plt.ylabel('Count')
        plt.show()

    # If a categorical group field exists, boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we successfully loaded and explored a Croissant-format dataset using `mlcroissant`. We examined available record sets and their fields, extracted records into DataFrames, conducted basic exploratory data analysis including filtering and normalization, and generated data visualizations.

For deeper insights, consider extending the exploration to more fields, additional EDA (correlation, advanced filtering), or integrating ML models using the processed data.